### Guardrails

Guardrails are safety mechanisms that control what foes into and comes out of an AI agent. They sit around your agent pipeline and ensure the agent:
1. only processes safe and appropriate inputs
2. only performs approved actions
3. only returns validated,, compliant outputs

Guardrails help you build safe, compliant AI applications by validating and filtering content at key points in your agent's execution.

They are implemented as *middleware* that intercepts execution:

- Before the agent starts (input guardrails)
- After it completes (output guardrails)
- Around model and tool calls

Common Use Cases:
1. PII leakage prevention -> Redact emails/credit cards before logging
2. Prompt injection blocking -> Detect adversarial inputs
3. Harmful content filtering -> Block dangerous requests
4. Business rule enforcement -> Require approval for financial ops
5. Output quality validation -> Ensure response meets safety standards



In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
import os
from getpass import getpass

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("GROQ API key loaded:", "✅" if GROQ_API_KEY    else "❌ Missing!")
print("OPENAI_API_KEY preview:", GROQ_API_KEY[:8] + "..." if GROQ_API_KEY else None)

GROQ API key loaded: ✅
OPENAI_API_KEY preview: gsk_OVh1...


### Section 2: Two Approaches to Guardrails

### Deterministic Guardrails
- Rule-based: regex, keyword matching, explicit checks
- ✅ Fast, predictable, cost-effective
- ❌ May miss nuanced violations

### Model-Based Guardrails
- Uses LLMs/classifiers for semantic understanding
- ✅ Catches subtle/nuanced issues
- ❌ Slower and more expensive

In [3]:
import re


def deterministic_guardrail(text:str) -> bool:
    """ Returns true if content is blocked"""
    banned_keywords = ['hack', 'exploit', 'malware', 'bomb']
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]

print("=== Deterministic Guardrail Demo ===")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "🚫 BLOCKED" if blocked else "✅ ALLOWED"
    print(f"{status}: {inp}")

=== Deterministic Guardrail Demo ===
🚫 BLOCKED: How do I hack into a database?
✅ ALLOWED: What is the capital of France?
🚫 BLOCKED: Explain how malware spreads


In [12]:
from langchain_groq import ChatGroq

## model based approach

def model_based_guardrail(text:str) -> str:
    """ Uses and LLM to evaluate content safety. Returns SAFE or UNSAFE """
    model = ChatGroq(model="groq:llama-3.1-8b-instant", temperature=0.0)
    prompt=f"""Is the following user input safe to process? 
    Reply with only 'SAFE' or 'UNSAFE'
    Input: {text}"""
    
    result = model.invoke([{"role": "user", "content": prompt}])
    return result.content.strip()

print("=== Model-Based Guardrail Demo ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "🚫 UNSAFE" if "UNSAFE" in verdict else "✅ SAFE"
    print(f"{status}: {inp}")

=== Model-Based Guardrail Demo ===


NotFoundError: Error code: 404 - {'error': {'message': 'The model `groq:llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

### Section 3: Built-in Guardrail — PII Detection Middleware
LangChain provides built-in PIIMiddleware for detecting and handling Personally Identifiable Information (PII).

#### Supported PII Types:

1. email -> user@example.com
2. credit_card -> 5105-1051-0510-5100
3. ip -> 192.168.1.1
4. mac_address	-> 00:1A:2B:3C:4D:5E
5. url	-> https://secret-site.com
#### Strategies:
1. redact -> [REDACTED_EMAIL]
2. mask -> ****-****-****-1234
3. hash -> a8f5f167...
4. block -> Raises an exception

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_groq import ChatGroq
from langchain_core.tools import tool

@tool #define dummy tool
def customer_lookup(query: str) -> str:
    """ Look up customer information"""
    return f"Customer record found for query: {query}"

## Create agent with PII Middleware
agent = create_agent(
    model="llama-3.1-8b-instant",
    tools=[customer_lookup],
    middleware=[
        ## Redact email in user input before sending to the model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True
        ),
        ## Mask credit cards in user input
        PIIMiddleware(
            'credit_card',
            strategy='mask',
            apply_to_input=True
        ),
        ## Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True
        )
    ]
)

print("Agent wih PII middleware created successfully")


# Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)


# === Agent Response ===
# I found the customer record associated with the card ending in 5100. How may I assist you further with this information?

print(result)


# {'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='0ee06704-0db8-4761-8a8c-384ab33c3203'),
#   AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 70, 'total_tokens': 91, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_dbff050aca', 'id': 'chatcmpl-DFwY2MLKqMx8KTqaQVaoGt6qPIrk4', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cbcb2-a5d6-71a2-ad11-d66a36788df9-0', tool_calls=[{'name': 'customer_lookup', 'args': {'query': '****-****-****-5100'}, 'id': 'call_OramGIuVaxnSsOIxTyQpLIrQ', 'type': 'tool_call'}], usage_metadata={'input_tokens': 70, 'output_tokens': 21, 'total_tokens': 91, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
#   ToolMessage(content='Customer record found for query: ****-****-****-5100', name='customer_lookup', id='ec657666-837e-41dc-bac5-993a25dd4baf', tool_call_id='call_OramGIuVaxnSsOIxTyQpLIrQ'),
#   AIMessage(content='I found the customer record associated with the card ending in 5100. How may I assist you further with this information?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 113, 'total_tokens': 139, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_dbff050aca', 'id': 'chatcmpl-DFwY3lDUQwPi5EsZxbUJzWIwoaCyJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cbcb2-a9b5-7f60-aa73-421d31cd289d-0', usage_metadata={'input_tokens': 113, 'output_tokens': 26, 'total_tokens': 139, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}

# Test API Key Blocking
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
    
except Exception as e:
    print(f"🚫 Blocked as expected: {e}")
       
# 🚫 Blocked as expected: Detected 1 instance(s) of api_key in text content

### Section 4: Built-in Guardrail — Human-in-the-Loop Middleware
Pauses agent execution before sensitive operations and waits for human approval.

##### Best for:

1. Financial transactions
2. Sending emails to external parties
3. Deleting production data
4. Any operation with significant business impact
5. Key requirement: A checkpointer for state persistence across interrupts.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

## Create agent with HITL middleware
hitl_agent = create_agent(
    model="llama-3.1-8b-instant",
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                'send_email': True,
                'delete_records': True,
                'search_web': True
            }
        )
    ],
    checkpointer=InMemorySaver() # Requiredfor state persistence
)

print('Human-in-the-loop Middleware created')

# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)


# === Agent paused — awaiting human approval ===
# {'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='1a7ba4e5-186c-4b98-9dcc-57013ef0e2bc'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 109, 'total_tokens': 179, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_54761c4e64', 'id': 'chatcmpl-DFwgQxpLLTTLBwdzMFV7mkr7IPSRu', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cbcba-9300-7c31-abda-55b2a56ac799-0', tool_calls=[{'name': 'send_email', 'args': {'to': 'team@company.com', 'subject': 'Q4 Results', 'body': 'Hello Team,\n\nI hope this message finds you well. I am writing to share the results for Q4. Please review the attached documentation for detailed insights.\n\nBest regards,\n\n[Your Name]'}, 'id': 'call_F70y6jaoPpXgeYUsT7ji5Hny', 'type': 'tool_call'}], usage_metadata={'input_tokens': 109, 'output_tokens': 70, 'total_tokens': 179, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})], '__interrupt__': [Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'to': 'team@company.com', 'subject': 'Q4 Results', 'body': 'Hello Team,\n\nI hope this message finds you well. I am writing to share the results for Q4. Please review the attached documentation for detailed insights.\n\nBest regards,\n\n[Your Name]'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'to': 'team@company.com', 'subject': 'Q4 Results', 'body': 'Hello Team,\\n\\nI hope this message finds you well. I am writing to share the results for Q4. Please review the attached documentation for detailed insights.\\n\\nBest regards,\\n\\n[Your Name]'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='4ba0fef56c2d02af01cb2022f4ae0596')]}


# Step 2: Human reviews and APPROVES
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config   # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)
# === Approved! Final response ===
# I've sent the email to team@company.com about the Q4 results.

# Step 3: Alternative — Human REJECTS
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)
# === Rejected! Final response ===
# It seems that you've decided not to proceed with the deletion. If you have any questions or need further assistance, feel free to ask!

### Section 5: Custom Guardrail — Before-Agent Hook (Input Filter)
Use before_agent() to validate or block requests before any LLM processing begins.

Best for:

- Keyword/content filtering
- Authentication checks
- Rate limiting
- Blocking specific categories of requests

In [ ]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"🚫 Blocked — keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
        return None


@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


# Create agent with content filter
filtered_agent = create_agent(
    model="gpt-4o",
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

print("Content filter agent created!")


# Content filter agent created!

# Test 1: Safe request — should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("✅ Safe request response:")
print(result["messages"][-1].content)


# ✅ Safe request response:
# Machine learning is a branch of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit instructions. Instead, these systems learn from patterns and inferences in data. Here are some key aspects of machine learning:

# 1. **Data-Driven:** Machine learning relies heavily on data. The algorithms use this data to learn and make predictions or decisions without being explicitly programmed for the task.

# 2. **Model Training:** The process involves training a model on a set of input data and using this model to predict or classify new data.

# 3. **Types of Learning:**
#    - **Supervised Learning:** The model is trained on labeled data. It learns to map inputs to outputs based on example input-output pairs.
#    - **Unsupervised Learning:** The model is trained on data without labeled responses and must infer the natural structure within a set of data points.
#    - **Reinforcement Learning:** The model learns by interacting with its environment and receiving rewards or penalties.

# 4. **Applications:** Machine learning is used in a variety of applications, including image and speech recognition, recommendation systems, medical diagnosis, stock trading, and more.

# 5. **Algorithms:** Common algorithms used in machine learning include decision trees, neural networks, support vector machines, and others.

# Machine learning allows for improved decision-making by learning from data and adapting to new circumstances or changes in data patterns.


# Test 2: Unsafe request — should be blocked
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})
print("🚫 Unsafe request response:")
print(result["messages"][-1].content)

# 🚫 Blocked — keyword detected: 'hack'
# 🚫 Unsafe request response:
# I cannot process requests containing inappropriate content. Please rephrase your request.


### Section 6: Custom Guardrail — After-Agent Hook (Output Safety)
Use after_agent() to validate the final agent response before the user sees it.

Best for:

- Model-based safety evaluation of outputs
- Compliance scanning (e.g. legal, medical, financial disclaimers)
- Quality validation
- Removing sensitive info that slipped through

In [ ]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool

class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a lightweight model as the safety judge
        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content.upper():
            print("⚠️  Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None


@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"


safe_agent = create_agent(
    model="gpt-4o",
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

print("Output safety agent created!")
# Output safety agent created!

# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "What is the weather like today?"}]
})
print("Response:")
print(result["messages"][-1].content)


# Response:
# I fetched today's weather forecast for you. Please let me know if there's anything specific you would like to know about the weather!

### Section 7: Layered / Combined Guardrails
Stack multiple guardrails in the middleware=[] array. They execute in order, building layered protection.

User Input
    
    ↓

[Layer 1] ContentFilterMiddleware    ← Deterministic input filter
    
    ↓

[Layer 2] PIIMiddleware (input)      ← PII redaction on input
   
    ↓

[Layer 3] HumanInTheLoopMiddleware   ← Approval for sensitive tools
   
    ↓

[Layer 4] PIIMiddleware (output)     ← PII redaction on output
   
    ↓

[Layer 5] SafetyGuardrailMiddleware  ← Model-based output safety
   
    ↓

User Response

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"

## Full layered guardrail stack

production_agent = create_agent(
    model="gpt-4o",
    tools=[search_tool, send_email_tool],
    middleware=[
        # layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=['hack', 'exploit', 'malware']),
        
        #Layer 2: PII redaction on input
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        
        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool": True, "search_tool": False}
        ),

        # Layer 4: PII redaction on output
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 5: Model-based output safety
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver()
)

print("🏭 Production-grade agent with 5-layer guardrails created!")


### Section 8: Real-World Use Case — Healthcare Chatbot
A healthcare chatbot that:

1. Blocks off-topic or harmful requests
2. Redacts patient PII (emails, credit card numbers)
3. Requires human approval before booking appointments
4. Validates that outputs are medically appropriate

In [ ]:
from langchain.agents.factory import _get_can_jump_to
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage

## Healthcare specific content filter

class HealthCareSafetyFilter(AgentMiddleware):
    """ Block non-medical or harmful requests in a healthcare context. """
    
    BLOCKED_TOPICS = ['drug syntesis', 'self harm', 'suicide method', 'weapon', 'hack']
    
    hook_config(_get_can_jump_to=['end'])
    def before_agent(self, state:AgentState, runtime:Runtime) -> dict[str, Any] | None:
        if not state['messages']:
            return None
        
        first_msg = state['messages'][0]
        if first_msg.type != 'human':
            return None
        
        content = first_msg.content.lower()
        for topic in self.BLOCKED_TOPICS:
            if topic in content:
                return {
                    'messages' : [{
                        'role':"assistant",
                        'content': (
                            "I'm a healthcare assistant and can only help with "
                            "medical questions, appointments, and health information. "
                            "If you are in crisis, please call 112 or your local emergency number"
                        )
                    }],
                    'jump_to':'end'
                }
        return None
    
## Medical output validator

class MedicalOutputValidator(AgentMiddleware):
    """ Ensure all responses include appropriate medical disclaimers. """
    
    DISCLAIMER = '\n\n  *This is a general health information, not medical advice. Please consult a qualified healthcare professional.* '
    
    @hook_config(_get_can_jump_to=['end'])
    def after_agent(self, state:AgentState, runtime:Runtime) -> dict[str, Any] | None:
        if not state['messages']:
            return None
        
        last_msg = state['messages']
        if not isinstance(last_msg, AIMessage):
            return None
        
        ## Add disclaimer if not already present
        if "medical_advice" not in last_msg.content.lower():
            last_msg.content += self.DISCLAIMER
            
        return None
    

## Healthcare tools 

@tool
def search_symptoms(symptoms: str) -> str:
    """ Search for information about medical symptoms """
    return f"Symptom information for: {symptoms}. Please consult a doctor for diagnosis"

@tool 
def book_appointment(patient_name: str, date: str, doctor: str) -> str:
    """ Book a medical appointment """
    return f"Appointmnet booked for {patient_name} with Dr. {doctor} on {date}" 

@tool
def get_medication_information(medication:str) -> str:
    """ Get information about a medication """
    return f"General info about {medication}. Always follow your doctor's prescription" 

## Build healthcare chatbot
healthcare_bot = create_agent(
    model = 'gpt-4o',
    tools=[search_symptoms, book_appointment, get_medication_information],
    middleware=[
        ## Guardrail1 : block harmful/off topic requests
        HealthCareSafetyFilter(),
        
        ## Guardrail 2 : Redact patient PII from inputs
        PIIMiddleware('email', strategy='redact', apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Guardrail 3: Require approval before booking appointments
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_appointment": True,
                "search_symptoms": False,
                "get_medication_info": False,
            }
        ),

        # Guardrail 4: Add medical disclaimer to all outputs
        MedicalOutputValidator(),
    ],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "You are a helpful healthcare assistant. "
        "You can search for symptoms, medication information, and help book appointments. "
        "Always be empathetic and remind users to consult a doctor for diagnosis."
    )
)

print("🏥 Healthcare chatbot with full guardrail stack created!")

# Test 1: Safe medical query
config_t1 = {"configurable": {"thread_id": "healthcare_session_t1"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "What are symptoms of Type 2 Diabetes?"}]},
    config=config_t1
)

result

# {'messages': [HumanMessage(content='What are symptoms of Type 2 Diabetes?', additional_kwargs={}, response_metadata={}, id='4b3fd6e6-a043-4367-8a6b-4661d7b3a6aa'),
#   AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 140, 'total_tokens': 159, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_64dfa806c7', 'id': 'chatcmpl-DFvdcvPema5HLJjSsrsLg7TCimODt', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cbc7d-4311-7751-b828-a13797954643-0', tool_calls=[{'name': 'search_symptoms', 'args': {'symptoms': 'Type 2 Diabetes'}, 'id': 'call_LElUXrARIRsGoVAEowpMGfmv', 'type': 'tool_call'}], usage_metadata={'input_tokens': 140, 'output_tokens': 19, 'total_tokens': 159, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
#   ToolMessage(content='Symptom information for: Type 2 Diabetes. Please consult a doctor for diagnosis.', name='search_symptoms', id='0869aa4a-9d88-4efc-8572-757f54b22112', tool_call_id='call_LElUXrARIRsGoVAEowpMGfmv'),
#   AIMessage(content="Here is some information about the symptoms of Type 2 Diabetes. However, it's important to consult with a doctor for any diagnosis:\n\nCommon symptoms of Type 2 Diabetes include:\n\n1. Increased thirst\n2. Frequent urination, especially at night\n3. Increased hunger\n4. Unplanned weight loss\n5. Fatigue or tiredness\n6. Blurred vision\n7. Slow-healing sores or frequent infections\n8. Areas of darkened skin, usually in the armpits and neck\n\nIf you're experiencing any of these symptoms, please reach out to a healthcare professional for a proper assessment.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 124, 'prompt_tokens': 185, 'total_tokens': 309, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_2cf39eccd5', 'id': 'chatcmpl-DFvddJvvf6Y0IMb3xUisqd5VSjwXs', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cbc7d-469c-7b63-881e-6a5249fef103-0', usage_metadata={'input_tokens': 185, 'output_tokens': 124, 'total_tokens': 309, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
#   HumanMessage(content='What are symptoms of Type 2 Diabetes?', additional_kwargs={}, response_metadata={}, id='4792eeb1-4e4d-4102-a7b4-37e0de3c853e'),
#   AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 325, 'total_tokens': 344, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_2cf39eccd5', 'id': 'chatcmpl-DFvdrByAE7rj3vnjEDrA9R8pAZovB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cbc7d-7f5b-7c52-8122-1a5c22f74a7c-0', tool_calls=[{'name': 'search_symptoms', 'args': {'symptoms': 'Type 2 Diabetes'}, 'id': 'call_tBvjtaiSkdpvDrjyXz4cwtjL', 'type': 'tool_call'}], usage_metadata={'input_tokens': 325, 'output_tokens': 19, 'total_tokens': 344, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
#   ToolMessage(content='Symptom information for: Type 2 Diabetes. Please consult a doctor for diagnosis.', name='search_symptoms', id='47d373eb-9374-4794-a68e-3d7f585dbd5e', tool_call_id='call_tBvjtaiSkdpvDrjyXz4cwtjL'),
#   AIMessage(content="Here is some information about the symptoms of Type 2 Diabetes. However, it's important to consult with a doctor for any diagnosis:\n\nCommon symptoms of Type 2 Diabetes include:\n\n1. Increased thirst\n2. Frequent urination, especially at night\n3. Increased hunger\n4. Unplanned weight loss\n5. Fatigue or tiredness\n6. Blurred vision\n7. Slow-healing sores or frequent infections\n8. Areas of darkened skin, usually in the armpits and neck\n\nIf you're experiencing any of these symptoms, it's crucial to consult with a healthcare professional for a proper assessment and diagnosis.\n\n⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 127, 'prompt_tokens': 370, 'total_tokens': 497, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_2cf39eccd5', 'id': 'chatcmpl-DFvdsA9iTf3m23lkMEMjj1yMTrbJ7', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cbc7d-8372-75f1-b02b-9309e161a6b4-0', usage_metadata={'input_tokens': 370, 'output_tokens': 127, 'total_tokens': 497, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}

# Test 2: Query with PII (email gets redacted)
result = healthcare_bot.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is patient123@gmail.com. What can I take for a headache?"
    }]},
    config=config_t1
)
print("=== PII Redaction Test ===")
print(result["messages"][-1].content)

# === PII Redaction Test ===
# I'm sorry, but I can't assist with using your email address or personal contact information. For a headache, there are several over-the-counter medications that might help, such as:

# 1. **Acetaminophen (Tylenol)**
# 2. **Ibuprofen (Advil, Motrin)**
# 3. **Aspirin**
# 4. **Naproxen (Aleve)**

# However, it's essential to follow the dosage instructions on the packaging and consider any personal health conditions or medications you may already be taking. If you have frequent headaches or your headache persists, it's best to consult a healthcare provider for advice tailored to your specific needs.

# ⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*


# Test 3: Off-topic / harmful request — gets blocked
result = healthcare_bot.invoke({
    "messages": [{"role": "user", "content": "How do I synthesize drugs at home?"}]
},
 config=config_t1)
print("=== Blocked Request ===")
print(result["messages"][-1].content)

# === Blocked Request ===
# I'm sorry, but I can't assist with that. It's important to note that synthesizing drugs at home is extremely dangerous, illegal, and poses significant health risks. Medications should only be manufactured by licensed professionals in controlled settings to ensure their safety and efficacy.

# If you have questions about medications or if you're seeking treatment options, please consult with a healthcare professional. They can provide guidance that's both safe and legal.

# ⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*


# Test 4: Appointment booking — requires human approval
config = {"configurable": {"thread_id": "healthcare_session_001"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "Book me an appointment with Dr. Sharma on March 15"}]},
    config=config
)
print("=== Appointment Booking — Awaiting Approval ===")
print(result)

# Approve
from langgraph.types import Command
approved = healthcare_bot.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config
)
print("\n=== After Approval ===")
print(approved["messages"][-1].content)


# === Appointment Booking — Awaiting Approval ===
# {'messages': [HumanMessage(content='Book me an appointment with Dr. Sharma on March 15', additional_kwargs={}, response_metadata={}, id='0ebea729-0ad8-45c2-b38e-5a8e50d190bb'), AIMessage(content='Could you please provide your full name for the appointment booking with Dr. Sharma on March 15?\n\n⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 143, 'total_tokens': 164, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_2cf39eccd5', 'id': 'chatcmpl-DFvfpPIYdJGDzGJ7jtJcPZXZJhJbN', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cbc7f-5a32-7683-ab00-6ebbf757088d-0', usage_metadata={'input_tokens': 143, 'output_tokens': 21, 'total_tokens': 164, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}

# === After Approval ===
# Could you please provide your full name for the appointment booking with Dr. Sharma on March 15?

Key Takeaways
1. Guardrails = Middleware — implement them via the middleware=[] parameter in create_agent()
2. Layer your guardrails — defense in depth is best practice
3. Deterministic first, model-based second — use cheap rule-based checks early to avoid expensive LLM calls
4. Human-in-the-Loop requires a checkpointer — use InMemorySaver for dev, persistent store for production
5. Custom middleware gives you full control via before_agent() and after_agent() hooks

| Guardrail Type | Hook | When it Runs | Best For |
|---|---|---|---|
| **PII Middleware** | Input/Output | Around model calls | Data privacy, compliance |
| **Human-in-the-Loop** | Tool level | Before sensitive tools | High-stakes decisions |
| **Content Filter** | `before_agent` | Start of invocation | Blocking bad inputs early |
| **Safety Validator** | `after_agent` | End of invocation | Output quality/safety |
| **Custom Logic** | Any hook | Anywhere | Any business rule |